<a href="https://colab.research.google.com/github/Junhojuno/pytorch-tutorial/blob/master/Make_VGG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [0]:
import torch.nn as nn
import torch.utils.model_zoo as model_zoo

In [0]:
__all__ = [
    'VGG', 'vgg11', 'vgg11_bn', 'vgg13', 'vgg13_bn', 'vgg16', 'vgg16_bn',
    'vgg19_bn', 'vgg19',
]


# imagenet 데이터를 pre-training시켜놓은 모델
model_urls = {
    'vgg11': 'https://download.pytorch.org/models/vgg11-bbd30ac9.pth',
    'vgg13': 'https://download.pytorch.org/models/vgg13-c768596a.pth',
    'vgg16': 'https://download.pytorch.org/models/vgg16-397923af.pth',
    'vgg19': 'https://download.pytorch.org/models/vgg19-dcbb9e9d.pth',
    'vgg11_bn': 'https://download.pytorch.org/models/vgg11_bn-6002323d.pth',
    'vgg13_bn': 'https://download.pytorch.org/models/vgg13_bn-abd245e5.pth',
    'vgg16_bn': 'https://download.pytorch.org/models/vgg16_bn-6c64b313.pth',
    'vgg19_bn': 'https://download.pytorch.org/models/vgg19_bn-c79401a0.pth',
}

In [0]:
class VGG(nn.Module):

    def __init__(self, features, num_classes=1000, init_weights=True):
        super(VGG, self).__init__()
        self.features = features # couvolution layer를 의미한다.
        self.avgpool = nn.AdaptiveAvgPool2d((7, 7))
       
        # self.classifier가 fully connected layer를 의미
        self.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, num_classes),
        )
        if init_weights:
            self._initialize_weights()

    def forward(self, x):
        x = self.features(x) # convolution layer
        x = self.avgpool(x) # average pooling
        x = x.view(x.size(0), -1)
        x = self.classifier(x) # FC layer
        return x

    def _initialize_weights(self):
        for m in self.modules(): # self.modules는 뭐지? nn.Module에 있는건가?
            if isinstance(m, nn.Conv2d): # 만약 m이 nn.Conv2d일때를 의미
                # kaiming normalization은 activation function에 따라 초기화를 잘해줄수 있다?
                # activation function에 따라 최적의 초기화방식이 다른가 보다?!
                # 여기선 relu
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0) # bias는 0으로 세팅
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1) # batch norm의 weight는 전부 1
                nn.init.constant_(m.bias, 0) # batch norm의 bias는 전부 0
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)


In [8]:
# feature를 어떻게 만들어서 넣어줄 것이냐
# 예를 들어 cfg['A']로 진행 
#  'A': [64, 'M', 128, 'M', 256, 256, 'M', 512, 512, 'M', 512, 512, 'M']

def make_layers(cfg, batch_norm=False):
    layers = []
    in_channels = 3 
    for v in cfg:
        if v == 'M': # v는 처음이 64, 맨 마지막이 'M' 이겠지?! yes!
            layers += [nn.MaxPool2d(kernel_size=2, stride=2)]
        else:
            conv2d = nn.Conv2d(in_channels, v, kernel_size=3, padding=1)
            if batch_norm:
                layers += [conv2d, nn.BatchNorm2d(v), nn.ReLU(inplace=True)]
            else:
                layers += [conv2d, nn.ReLU(inplace=True)]
            in_channels = v
    return nn.Sequential(*layers)
  
"""
model = make_layers(cfg['A'], batch_norm=False)로 했을때 
어떤 모델이 생성될지를 확인해보자
아래가 순서대로 nn.Sequential에 들어간다.

<첫번째. 64가 들어갔을때>
conv2d = nn.Conv2d(3, 64, kernel_size=3, padding=1)
nn.ReLU(inplace=True)

<두번째. 'M'>
nn.MaxPool2d(kernel_size=2, stride=2)

<세번째. 128>
conv2d = nn.Conv2d(64, 128, kernel_size=3, padding=1)
nn.ReLU(inplace=True)

<네번째. 'M'>
nn.MaxPool2d(kernel_size=2, stride=2)

<다섯번째. 256>
conv2d = nn.Conv2d(128, 256, kernel_size=3, padding=1)
nn.ReLU(inplace=True)

<여섯번째. 256>
conv2d = nn.Conv2d(256, 256, kernel_size=3, padding=1)
nn.ReLU(inplace=True)

<일곱번째. 'M'>
nn.MaxPool2d(kernel_size=2, stride=2)

<여덟번째. 512>
conv2d = nn.Conv2d(256, 512, kernel_size=3, padding=1)
nn.ReLU(inplace=True)

<아홉번째. 512>
conv2d = nn.Conv2d(512, 512, kernel_size=3, padding=1)
nn.ReLU(inplace=True)

<열번째. 'M'>
nn.MaxPool2d(kernel_size=2, stride=2)

<열한번째. 512>
conv2d = nn.Conv2d(512, 512, kernel_size=3, padding=1)
nn.ReLU(inplace=True)

<열두번째. 512>
conv2d = nn.Conv2d(512, 512, kernel_size=3, padding=1)
nn.ReLU(inplace=True)

<열세번째. 'M'>
nn.MaxPool2d(kernel_size=2, stride=2)

<return은...>
# convolution layer 8개
nn.Sequential(
              nn.Conv2d(3, 64, kernel_size=3, padding=1) # 1
             ,nn.ReLU(inplace=True)
             ,nn.MaxPool2d(kernel_size=2, stride=2)
             ,nn.Conv2d(64, 128, kernel_size=3, padding=1) # 2
             ,nn.ReLU(inplace=True)
             ,nn.MaxPool2d(kernel_size=2, stride=2)
             ,nn.Conv2d(128, 256, kernel_size=3, padding=1) # 3
             ,nn.ReLU(inplace=True)
             ,nn.Conv2d(256, 256, kernel_size=3, padding=1) # 4
             ,nn.ReLU(inplace=True)
             ,nn.MaxPool2d(kernel_size=2, stride=2)
             ,nn.Conv2d(256, 512, kernel_size=3, padding=1) # 5
             ,nn.ReLU(inplace=True)
             ,nn.Conv2d(512, 512, kernel_size=3, padding=1) # 6
             ,nn.ReLU(inplace=True)
             ,nn.MaxPool2d(kernel_size=2, stride=2)
             ,nn.Conv2d(512, 512, kernel_size=3, padding=1) # 7
             ,nn.ReLU(inplace=True)
             ,nn.Conv2d(512, 512, kernel_size=3, padding=1) # 8
             ,nn.ReLU(inplace=True)
             ,nn.MaxPool2d(kernel_size=2, stride=2)
)

"""

"\nmodel = make_layers(cfg['A'], batch_norm=False)로 했을때 \n어떤 모델이 생성될지를 확인해보자\n아래가 순서대로 nn.Sequential에 들어간다.\n\n<첫번째. 64가 들어갔을때>\nconv2d = nn.Conv2d(3, 64, kernel_size=3, padding=1)\nnn.ReLU(inplace=True)\n\n<두번째. 'M'>\nnn.MaxPool2d(kernel_size=2, stride=2)\n\n<세번째. 128>\nconv2d = nn.Conv2d(64, 128, kernel_size=3, padding=1)\nnn.ReLU(inplace=True)\n\n<네번째. 'M'>\nnn.MaxPool2d(kernel_size=2, stride=2)\n\n<다섯번째. 256>\nconv2d = nn.Conv2d(128, 256, kernel_size=3, padding=1)\nnn.ReLU(inplace=True)\n\n<여섯번째. 256>\nconv2d = nn.Conv2d(256, 256, kernel_size=3, padding=1)\nnn.ReLU(inplace=True)\n\n<일곱번째. 'M'>\nnn.MaxPool2d(kernel_size=2, stride=2)\n\n<여덟번째. 512>\nconv2d = nn.Conv2d(256, 512, kernel_size=3, padding=1)\nnn.ReLU(inplace=True)\n\n<아홉번째. 512>\nconv2d = nn.Conv2d(512, 512, kernel_size=3, padding=1)\nnn.ReLU(inplace=True)\n\n<열번째. 'M'>\nnn.MaxPool2d(kernel_size=2, stride=2)\n\n<열한번째. 512>\nconv2d = nn.Conv2d(512, 512, kernel_size=3, padding=1)\nnn.ReLU(inplace=True)\n\n<열두번째. 512>\nconv2

In [0]:
# 'custom'추가도 가능하다.
cfg = {
    'A': [64, 'M', 128, 'M', 256, 256, 'M', 512, 512, 'M', 512, 512, 'M'], # convolution 8 + fc 3 = vgg11
    'B': [64, 64, 'M', 128, 128, 'M', 256, 256, 'M', 512, 512, 'M', 512, 512, 'M'], # 10 + 3 = vgg13
    'D': [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 'M', 512, 512, 512, 'M', 512, 512, 512, 'M'], # 13 + 3 = vgg16
    'E': [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 256, 'M', 512, 512, 512, 512, 'M', 512, 512, 512, 512, 'M'], # 16 + 3 = vgg19
    'custom' : [64,64,64,'M',128,128,128,'M',256,256,256,'M']
}

In [9]:
# 비교해보자
# 직접 계산한거랑 맞는지
conv = make_layers(cfg=cfg['A'])
conv

Sequential(
  (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU(inplace)
  (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (4): ReLU(inplace)
  (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (7): ReLU(inplace)
  (8): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (9): ReLU(inplace)
  (10): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (11): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (12): ReLU(inplace)
  (13): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (14): ReLU(inplace)
  (15): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (16): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (17)

In [10]:
# batch_norm=True로 놓으면
# 각 convolution아래 batch norm이 추가된 것을 볼 수 있다.
conv2 = make_layers(cfg['A'], batch_norm=True)
conv2

Sequential(
  (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace)
  (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (4): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (5): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (6): ReLU(inplace)
  (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (8): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (9): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (10): ReLU(inplace)
  (11): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (12): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (13): ReLU(inplace)
  (14): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (15)

In [12]:
# 실제 VGG를 만들어보자
# class 갯수는 cifar10인 경우 10개
# fc layer에서 받는 in_feature의 수가 고정되어있기 때문에
# custom VGG사용하려면 이부분을 적절하게 수정해줘야한다.
model = VGG(features=make_layers(cfg['A']), num_classes=10, init_weights=True)
model

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU(inplace)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace)
    (8): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace)
    (10): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (11): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): ReLU(inplace)
    (13): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (14): ReLU(inplace)
    (15): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (16): Conv2d(512, 512, kern